# Introduction to Density Matrices

Density matrices provide a more general framework for describing quantum states than statevectors. While statevectors represent **pure states**, density matrices can represent both **pure states** and **mixed states** (statistical mixtures).

## Topics Covered
- Pure vs Mixed States
- Outer Products and Density Matrix Definition
- Mathematical Properties (Trace, Hermiticity, Positivity)
- Purity and Von Neumann Entropy
- Fidelity and State Similarity
- Multi-qubit Systems


## 1. Setup & Authentication

We reuse the same authentication pattern as previous modules: load `API_KEY` from `qcloud.env` and authenticate with `QpiAIQuantumAuth`.

In [ ]:
import os

from dotenv import load_dotenv
from qpiai_quantum import QpiAIQuantumAuth

load_dotenv("./qcloud.env") # This path should point to the env file containing your API key.

QpiAIQuantumAuth.login(os.getenv("API_KEY"))
user_info = QpiAIQuantumAuth.me()

print(f"✅ Authenticated successfully as: {user_info.get('name', 'User')} ({user_info.get('email', '')})")

✅ Authenticated successfully as: Test Advanced User (test_advanced@qpiai.tech)


## 2. Why Do We Need Density Matrices?

In quantum computing, we often describe a qubit using a **statevector**:

$$\rvert \psi \rangle = \alpha \rvert 0 \rangle + \beta \rvert 1 \rangle$$

This works when we know exactly how the system was prepared, a situation known as a **pure state**. However, statevectors cannot represent **classical uncertainty**.

There are two fundamentally different types of uncertainty in quantum mechanics:

1. **Quantum Uncertainty (Superposition)**: Represented by statevectors. The system is in a linear combination of basis states.
2. **Classical Uncertainty (Mixed States)**: Represented by density matrices. We have a probabilistic mixture of different quantum states (e.g., due to noise, decoherence, or lack of knowledge about preparation).

### Classical Uncertainty Example

Suppose we flip a fair coin:
- Heads $\rightarrow$ prepare $\rvert 0 \rangle$
- Tails $\rightarrow$ prepare $\rvert 1 \rangle$

If we don't know the outcome, the system is **not** in a superposition $(\rvert 0 \rangle + \rvert 1 \rangle)/\sqrt{2}$. Instead, it is either $\rvert 0 \rangle$ or $\rvert 1 \rangle$ with 50% probability each. A single statevector cannot describe this "either-or" classical mixture. This is where density matrices are essential.

> **Note**: Quantum computing is structured linear algebra constrained by physical postulates. Master the structure, and everything else reduces to implementation.

## 3. Formal Definition

### Pure States
For a pure state $\rvert \psi \rangle$, the density matrix $\rho$ is defined as the **outer product**:

$$\rho = \rvert \psi \rangle \langle \psi \rvert$$

If $\rvert \psi \rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$, then:

$$\rho = \begin{pmatrix} \alpha \\ \beta \end{pmatrix} \begin{pmatrix} \alpha^* & \beta^* \end{pmatrix} = \begin{pmatrix} |\alpha|^2 & \alpha \beta^* \\ \alpha^* \beta & |\beta|^2 \end{pmatrix}$$

- **Diagonal entries**: Represent measurement probabilities in the computational basis.
- **Off-diagonal entries**: Represent **quantum coherence** (phase information).

### Mixed States
If a system is in state $\rvert \psi_i \rangle$ with probability $p_i$, its density matrix is a **statistical mixture**:

$$\rho = \sum_i p_i \rvert \psi_i \rangle \langle \psi_i \rvert$$

An example is the **maximally mixed state** for a single qubit (50% $\rvert 0 \rangle$ and 50% $\rvert 1 \rangle$):
$$\rho = 0.5 \rvert 0 \rangle \langle 0 \rvert + 0.5 \rvert 1 \rangle \langle 1 \rvert = \begin{pmatrix} 0.5 & 0 \\ 0 & 0.5 \end{pmatrix}$$

Note the lack of off-diagonal terms, indicating a complete loss of coherence.

In [2]:
import numpy as np
from qpiai_quantum import QuantumRegister, ClassicalRegister, Circuit
from qpiai_quantum.quantum_info import DensityMatrix


# Creating a pure state density matrix from a label
dm_pure = DensityMatrix.from_label("0")
print("Pure State |0><0|:\n", dm_pure._get_data())

# Creating a mixed state density matrix from a numpy array
rho_mixed = np.array([[0.5, 0], [0, 0.5]])
dm_mixed = DensityMatrix(rho_mixed)
print("\nMixed State (0.5|0><0| + 0.5|1><1|):\n", dm_mixed._get_data())

Pure State |0><0|:
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]

Mixed State (0.5|0><0| + 0.5|1><1|):
 [[0.5+0.j 0. +0.j]
 [0. +0.j 0.5+0.j]]


## 4. Mathematical Properties

A physical density matrix must satisfy three key conditions:

1. **Hermitian**: $\rho = \rho^\dagger$. This ensures all eigenvalues (probabilities) are real.
2. **Unit Trace**: $\mathrm{Tr}(\rho) = 1$. The sum of probabilities must be exactly 1.
3. **Positive Semidefinite**: $\rho \ge 0$. All eigenvalues must be non-negative (probabilities $\ge 0$).

### Purity Test
We can distinguish pure from mixed states using the **purity**: $\gamma = \mathrm{Tr}(\rho^2)$.
- **Pure state**: $\mathrm{Tr}(\rho^2) = 1$ (and $\rho^2 = \rho$)
- **Mixed state**: $\mathrm{Tr}(\rho^2) < 1$

In [3]:
print(f"Pure DM - Trace: {dm_pure.trace()}, Purity: {dm_pure.purity()}, Is Pure: {dm_pure.is_pure()}")
print(f"Mixed DM - Trace: {dm_mixed.trace()}, Purity: {dm_mixed.purity()}, Is Pure: {dm_mixed.is_pure()}")

print(f"Pure DM valid? {dm_pure.is_valid()}")
print(f"Mixed DM valid? {dm_mixed.is_valid()}")

Pure DM - Trace: (1+0j), Purity: 1.0, Is Pure: True
Mixed DM - Trace: (1+0j), Purity: 0.5, Is Pure: False
Pure DM valid? True
Mixed DM valid? True


## 5. Von Neumann Entropy

The **Von Neumann Entropy** $S(\rho)$ quantifies the uncertainty or "mixedness" of a quantum state:

$$S(\rho) = -\mathrm{Tr}(\rho \log_2 \rho) = -\sum_i \lambda_i \log_2 \lambda_i$$

where $\lambda_i$ are the eigenvalues of $\rho$.

- **Pure state**: $S(\rho) = 0$ (maximum knowledge, zero entropy).
- **Maximally mixed state**: $S(\rho) = \log_2 d$ (minimum knowledge, maximum entropy), where $d$ is the dimension.

In [4]:
print(f"Entropy of pure state: {dm_pure.von_neumann_entropy()}")
print(f"Entropy of maximally mixed state: {dm_mixed.von_neumann_entropy()}")

Entropy of pure state: -0.0
Entropy of maximally mixed state: 1.0


## 6. Fidelity

Fidelity measures the similarity between two quantum states $\rho$ and $\sigma$:

$$F(\rho, \sigma) = \left( \mathrm{Tr} \sqrt{\sqrt{\rho} \sigma \sqrt{\rho}} \right)^2$$

- **$F = 1$**: The states are identical.
- **$F = 0$**: The states are perfectly distinguishable (orthogonal).

### Special Case: Pure States
If both states are pure ($\rho = \rvert \psi \rangle \langle \psi \rvert$ and $\sigma = \rvert \phi \rangle \langle \phi \rvert$), fidelity simplifies to the squared overlap:
$$F(\rho, \sigma) = |\langle \psi \rvert \phi \rangle|^2$$

In [5]:
dm0 = DensityMatrix.from_label("0")
dm1 = DensityMatrix.from_label("1")

# Create |+> state through a circuit
circ_plus = Circuit(QuantumRegister(1))
circ_plus.h(0)
dm_plus = DensityMatrix.from_circuit_object(circ_plus)

dm_mixed = DensityMatrix(np.eye(2) / 2)

print(f"Fidelity(0, 0): {dm0.fidelity(dm0)}")
print(f"Fidelity(0, 1): {dm0.fidelity(dm1)}")
print(f"Fidelity(0, +): {dm0.fidelity(dm_plus)}")
print(f"Fidelity(0, mixed): {dm0.fidelity(dm_mixed)}")

Fidelity(0, 0): 1.0
Fidelity(0, 1): 0.0
Fidelity(0, +): 0.5000000000000001
Fidelity(0, mixed): 0.5000000000000001


## 7. Advanced Usage and Multi-Qubit Systems

Density matrices are particularly powerful for:
- **Modeling noise** and decoherence in realistic hardware.
- **Partial Trace**: Examining a subsystem of an entangled system.
- **Dimension**: For $n$ qubits, the density matrix $\rho$ has dimension $2^n \times 2^n$.

In [6]:
# Two-qubit Bell State preparation
qr = QuantumRegister(2)
circ = Circuit(qr)
circ.h(0)
circ.cx(0, 1)

dm_bell = DensityMatrix.from_circuit_object(circ)
print("Bell State Density Matrix Shape:", dm_bell._get_data().shape)
print("Is Pure?", dm_bell.is_pure())
print("Validity Check:", dm_bell.is_valid())

# Two-qubit Product State (independent Hadamards)
qr_prod = QuantumRegister(2)
circ_prod = Circuit(qr_prod)
circ_prod.h(0)
circ_prod.h(1)
dm_prod = DensityMatrix.from_circuit_object(circ_prod)
print("\nProduct State (|++>) Shape:", dm_prod._get_data().shape)
print("Is Pure?", dm_prod.is_pure())

Bell State Density Matrix Shape: (4, 4)
Is Pure? True
Validity Check: True

Product State (|++>) Shape: (4, 4)
Is Pure? True


## Summary
Density matrices are the universal language of quantum states. They capture both quantum superposition and classical probability, making them indispensable for studying real-world quantum systems, noise, and information theory concepts like entropy and fidelity.

---
**Thank you for learning with QpiAI!**

In [7]:
import qpiai_quantum
print(qpiai_quantum.__version__)

0.1.13
